# Lab 3: Hugging Face versus an inference engine

This notebook contains the same steps as `README.md`. You run the same model
in two ways, **Hugging Face `transformers`** and the **vLLM** inference
engine, and for each you run a **speed benchmark** and an **evaluation** on
your Lab 2 test set. Then you design one experiment of your own.

**Where to run it:** in a JupyterLab session **with a GPU** on the GPU side of
Roihu. If you only have a JupyterLab session without a GPU, run the commands
in an interactive GPU session in a terminal instead (README step 2); the
cells show exactly what to type.

Each `%%bash` cell starts a new shell, so every cell goes to the lab
directory first.

## 1. Settings

Set `TASK` to the config of your Lab 2 task. Without your own task, use
`../../Week-2/Lab-2/configs/roihu.json` (the small example).

In [ ]:
import os

os.environ["TASK"] = "../../Week-2/Lab-2/configs/my-task.json"
os.environ["HF_HOME"] = f"/scratch/project_2021893/{os.environ.get('USER', 'unknown')}/hf"
print(os.environ["TASK"])

## 2. Smoke test

Checks every script with fake models (no GPU). The last line should be
`Local smoke test completed.`

In [ ]:
%%bash
cd ~/LLM-Course-2026/Week-3/Lab-3
[ -z "$(type -t module)" ] && source /etc/profile.d/modules.sh 2>/dev/null
bash run_smoke_test.sh | tail -n 3

## 3. Hugging Face: speed and accuracy

The model runs inside this Python process with `model.generate()`.
"Concurrency 8" means one batch of 8 prompts.

In [ ]:
%%bash
cd ~/LLM-Course-2026/Week-3/Lab-3
[ -z "$(type -t module)" ] && source /etc/profile.d/modules.sh 2>/dev/null
module purge
module load python-pytorch/2.13
python3 benchmark.py --backend hf --output-dir results/hf
python3 score_task.py --backend hf --task-config $TASK --output-dir results/hf

## 4. vLLM: speed and accuracy

Start the server (it keeps running in the background), then send it the same
benchmark and the same test set. "Concurrency 8" means 8 requests at the same
time.

In [ ]:
%%bash
cd ~/LLM-Course-2026/Week-3/Lab-3
[ -z "$(type -t module)" ] && source /etc/profile.d/modules.sh 2>/dev/null
bash scripts/start_server.sh

In [ ]:
%%bash
cd ~/LLM-Course-2026/Week-3/Lab-3
[ -z "$(type -t module)" ] && source /etc/profile.d/modules.sh 2>/dev/null
python3 benchmark.py --backend server --output-dir results/vllm
python3 score_task.py --backend server --task-config $TASK --output-dir results/vllm

## 5. Compare

Which is faster for one request, and for 8 at a time? Is the accuracy the
same? Where do the answers differ?

In [ ]:
%%bash
cd ~/LLM-Course-2026/Week-3/Lab-3
[ -z "$(type -t module)" ] && source /etc/profile.d/modules.sh 2>/dev/null
python3 summarize_results.py results
python3 show_differences.py results/hf results/vllm

In [ ]:
# Tokens per second against concurrency, for every run in results/
import json
from pathlib import Path

import matplotlib.pyplot as plt

root = Path.home() / "LLM-Course-2026/Week-3/Lab-3/results"
for summary in sorted(root.glob("*/speed-summary.json")):
    if summary.parent.name == "smoke":
        continue
    levels = json.loads(summary.read_text())["levels"]
    plt.plot([l["concurrency"] for l in levels], [l["tokens_per_second"] for l in levels],
             marker="o", label=summary.parent.name)
plt.xscale("log", base=2)
plt.xlabel("requests at the same time")
plt.ylabel("output tokens per second (all requests)")
plt.legend()
plt.show()

## 6. Your own experiment

Choose **one** question and run it below. Write a table and a paragraph that
explain the result with the lecture's ideas (prefill and decode, batching,
KV cache, memory bandwidth).

- **A. Where is the knee?** More requests at the same time (example below).
- **B. Prompt length.** Write files with short and with long prompts (one per
  line) and pass them with `--prompts`. What happens to the time to first
  token?
- **C. Predict, then measure.** Estimate the maximum tokens per second for one
  user: memory bandwidth (about 4 TB/s) divided by the model size (about
  1.6 GB in bf16). Compare with your concurrency-1 number and explain the gap.
- **D. Your own workload.** Prompts and an answer length typical of your task
  (`--prompts`, `--max-tokens`). Does vLLM's advantage hold?

In [ ]:
%%bash
cd ~/LLM-Course-2026/Week-3/Lab-3
[ -z "$(type -t module)" ] && source /etc/profile.d/modules.sh 2>/dev/null
# Example for A. Change it for your own question.
python3 benchmark.py --backend server --output-dir results/vllm-knee --concurrency 1 2 4 8 16 32 64

In [ ]:
%%bash
cd ~/LLM-Course-2026/Week-3/Lab-3
[ -z "$(type -t module)" ] && source /etc/profile.d/modules.sh 2>/dev/null
# Example for A, Hugging Face side.
module purge
module load python-pytorch/2.13
python3 benchmark.py --backend hf --output-dir results/hf-knee --concurrency 1 4 16

Run the summary and the plot cells in step 5 again to see the new runs.

## 7. Stop the server

A running server keeps its GPU memory. Stop it when you are done.

In [ ]:
%%bash
cd ~/LLM-Course-2026/Week-3/Lab-3
[ -z "$(type -t module)" ] && source /etc/profile.d/modules.sh 2>/dev/null
bash scripts/stop_server.sh

## Optional: FP8 weights

Restart the server with 8-bit weights and measure again.

In [ ]:
%%bash
cd ~/LLM-Course-2026/Week-3/Lab-3
[ -z "$(type -t module)" ] && source /etc/profile.d/modules.sh 2>/dev/null
bash scripts/start_server.sh configs/model-fp8.json
python3 benchmark.py --backend server --config configs/model-fp8.json --output-dir results/vllm-fp8
python3 score_task.py --backend server --config configs/model-fp8.json --task-config $TASK --output-dir results/vllm-fp8
bash scripts/stop_server.sh

## Optional: JSON with and without a schema

Needs the bf16 server from step 4 running.

In [ ]:
%%bash
cd ~/LLM-Course-2026/Week-3/Lab-3
[ -z "$(type -t module)" ] && source /etc/profile.d/modules.sh 2>/dev/null
bash scripts/start_server.sh
python3 evaluate_structured.py --task-config $TASK --output-dir results/vllm-json
bash scripts/stop_server.sh